In [1]:
from groq import Groq
import os

api_key = "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"
client = Groq(
    api_key=api_key,
)


In [2]:
from sentence_transformers import SentenceTransformer, util

texts = ["When the CPU usage exceeds 90% and the memory usage exceeds 85% for 5 minutes, an emergency alarm is triggered.",
         "The power supply system conducted an emergency drill on Wednesday afternoon",
         "The power supply system conducted an emergency drill on Wednesday afternoon, triggering instantaneous current fluctuations when UPS or generator switching",
         "When the power supply system is conducting emergency drills, the instantaneous current fluctuations triggered by UPS or generator switching may cause false alarms in the power monitoring system.",
         "2023-12-25 02:00-04:00 Power supply maintenance is being carried out on room A, which may cause a momentary power outage"
        ]

def text2rules(text:str):
    res_rules = client.chat.completions.create(
        model="deepseek-r1-distill-llama-70b",
        messages=[
            {
                "role": "system",
                "content": """You are an expert in data center monitoring, focusing on accurately converting natural language descriptions into ProbLog rules.
                """,
            },
            {
                "role": "user",
                "content":f"""Convert the following operation and maintenance plan into Prolog suppression rules to prevent false positives:
                Input: {text}
                Output format:
                suppress(alarm type, location) :-
                time_window(start time, end time),
                affected_components(component list)."""
            },
        ],
    )

    rules = res_rules.choices[0].message.content
    # print(rules)

    _, _, after_r = rules.partition("</think>")
    rules = after_r.split('```prolog')[1].split('```')[0].strip()
    print("-----------------Lang_to_Rules-----------------")
    print(rules)
    return rules

def rules2text(rules:str):
    res_texts = client.chat.completions.create(
        model="deepseek-r1-distill-llama-70b",
        messages=[
            {
                "role": "system",
                "content": """You are an expert in data center monitoring, focusing on accurately converting ProbLog rules into natural language descriptions.
                Use the following fixed format to wrap the natural language descriptions: ```lang\n[text content]\n```
                Aside descriptions please do not generate other things, thanks""",
            },
            {
                "role": "user",
                "content": f"{rules}"
            },
        ],
    )

    natural_lang = res_texts.choices[0].message.content
    # print(natural_lang)

    _, _, after_t = natural_lang.partition("</think>")
    lang = after_t.split('```lang')[1].split('```')[0].strip()
    print("-----------------Rules_to_Lang-----------------")
    print(lang)
    return lang


model = SentenceTransformer('bert-base-nli-mean-tokens') # semantic validation model
for i, text in enumerate(texts):
    print(f"=================================ROUND{i+1}=====================================")
    print("origin text:", text)
    # text: original text / rules: generated problog rules / rev_text: generated text from rules
    rules = text2rules(text)
    rev_text = rules2text(rules)
    embeddings = model.encode([text, rules, rev_text])
    similarity_t2r = util.cos_sim(embeddings[0], embeddings[1]).item()
    similarity_r2t = util.cos_sim(embeddings[1], embeddings[2]).item()
    similarity_t2t = util.cos_sim(embeddings[0], embeddings[2]).item()
    print("-----------------Similarity-----------------")
    print("similarity text2rules:", similarity_t2r, "rules2text:", similarity_r2t, "text2text:", similarity_t2t)
    print("\n")

/Users/zhenzhili/miniforge3/envs/logllm/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


=================================ROUND1=====================================
origin text: When the CPU usage exceeds 90% and the memory usage exceeds 85% for 5 minutes, an emergency alarm is triggered.
-----------------Lang_to_Rules-----------------
suppress(emergency_alarm, data_center) :-
    time_window(StartTime, EndTime),
    cpu_usage(CPUUsage),
    CPUUsage > 90,
    memory_usage(MemoryUsage),
    MemoryUsage > 85,
    duration(StartTime, EndTime, 5 minutes),
    affected_components([cpu, memory]).
-----------------Rules_to_Lang-----------------
The emergency alarm for the data center should be suppressed if both CPU usage exceeds 90% and memory usage exceeds 85% for a continuous period of 5 minutes.
-----------------Similarity-----------------
similarity text2rules: 0.7397419810295105 rules2text: 0.7867808938026428 text2text: 0.8956876993179321


=================================ROUND2=====================================
origin text: The power supply system conducted an emerge

IndexError: list index out of range

In [3]:
answer = client.chat.completions.create(
    model="deepseek-r1-distill-llama-70b",
    messages=[
        {
            "role": "user",
            "content":"我目前正在一家数据中心做我的外部硕士论文, 题目为“神经符号网络异常检测在数据中心”，导师推荐使用已经训练好的大模型(例如GPT4), 我想请问在同时确保“LLM输入自然语言”和“神经符号网络检测异常”的前提下, 你是否有切实可行的建议可供我参考? 请反复复盘确保建议的可行性, 同时请使用你的深度思考为我解答, 非常感谢!"
        },
    ],
)

In [5]:
print(answer.choices[0].message.content)

<think>
嗯，我现在在做一个关于神经符号网络异常检测的外部硕士论文，导师建议我用已经训练好的大模型，比如GPT-4。我需要在确保输入是自然语言的同时，用神经符号网络来检测异常。那该怎么做呢？

首先，我需要理解什么是神经符号网络。神经符号网络（Neural-Symbolic Networks）结合了神经网络和符号推理的优势，可能用于处理需要逻辑推理的任务。异常检测则是识别数据中的异常或异常模式。

接下来，我得考虑如何将GPT-4与神经符号网络结合起来。GPT-4擅长处理自然语言，但异常检测可能需要结构化的数据处理。或许可以将数据转换为自然语言形式，然后输入GPT-4进行处理，再结合符号推理来检测异常。

比如，假设我有服务器日志数据，里面包含各种指标和状态信息。我可以将这些数据转换成自然语言描述，比如“CPU使用率高于80%”。然后，GPT-4可以处理这些描述，生成潜在的异常模式或异常原因。接着，神经符号网络可以分析这些模式，应用预定义的规则或逻辑来判断是否存在异常。

不过，这里可能会遇到一些问题。首先，数据转换为自然语言是否足够准确？如果转换不够好，可能会影响后续的分析。其次，如何将GPT-4的输出与符号推理部分无缝衔接？可能需要设计一个中间件来处理。

另一个想法是，使用GPT-4进行初步的异常检测，然后将结果输入到神经符号网络中进行进一步的验证。比如，GPT-4可以生成可能的异常候选，然后神经符号网络根据逻辑规则判断这些候选是否真的是异常。

或者，反过来，先用神经符号网络处理结构化数据，生成初步的异常标志，然后用GPT-4对这些标志进行自然语言描述，生成更易理解的异常报告。这样可以同时利用两者的优势。

我还需要考虑模型的可解释性。神经符号网络通常更容易解释，因为它基于明确的逻辑规则，而GPT-4的黑箱特性可能会影响结果的解释。因此，如何在两者之间找到平衡，确保结果既准确又可解释，是一个挑战。

另外，数据隐私和安全也是一个问题。使用大模型处理数据中心的数据，可能涉及敏感信息。需要确保在处理过程中数据是匿名化的，或者采用适当的安全措施来保护数据。

在实施过程中，可能需要进行大量的数据预处理和模型微调。比如，调整GPT-4的输出格式，使其更适合神经符号网络的输入需求。同时，可能需要设计一个反馈机制，根据检测结果反馈到GPT-4，优化其生成的内容。
